# TFT Full Run — Kaggle GPU (Phase 2)
Full 100-epoch train of the Temporal Fusion Transformer lap-time model.

**Before running:**
1. Notebook settings (right panel): **Accelerator = GPU (P100 or T4)**, **Internet = ON**.
2. Add the data: **+ Add Input** → upload `tft_full_data.zip` as a new Dataset (or attach an existing one).
   The copy cell below auto-finds every `laps_*_r*.parquet` under `/kaggle/input/`, so the dataset name doesn't matter.

Bar to beat: LightGBM green val 2.18s (era0) / test 2.14s (era1).

In [ ]:
# 1. Install pf + lightning. Use Kaggle's stock torch (cu128).
# IMPORTANT: set Accelerator = GPU T4 (NOT P100). cu128 torch has no sm_60 kernels,
# so it cannot run on P100 (sm_60) -> "no kernel image". T4 is sm_75 and works.
!pip -q install 'pytorch-forecasting==1.7.0' 'lightning==2.6.5' mlflow fastf1 pandera scipy
import torch, pytorch_forecasting as pf, lightning
print('pf', pf.__version__, '| lightning', lightning.__version__, '| torch', torch.__version__)
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0) in {(7,5),(8,0),(8,6),(9,0)}, \
    'Switch Accelerator to T4 — this GPU arch is not in the torch build.'

In [ ]:
# 2. Clone repo (or pull if already present) + add to path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
print('HEAD commit:')
!cd $REPO && git log --oneline -1

In [ ]:
# 3. Copy parquets from the attached Kaggle dataset into the repo's data/raw
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data zip as a Dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
seasons = sorted({n.split('_')[1] for n in copied})
print(f'{len(copied)} parquet files in data/raw | seasons: {seasons}')

In [ ]:
# 4. Full run (100 epochs, EarlyStopping on val_loss). mlflow file-store opt-in is set in code.
from src.models.lap_time.train_tft import main
main(fast=False)

In [ ]:
# 5. Locate the CPU-loadable artifact to download (Output tab → right panel)
import glob
for p in glob.glob(f'{REPO}/models/*.pt') + glob.glob(f'{REPO}/models/*.ckpt'):
    print(p)

In [ ]:
# 6. Show the breakdown + calibration reports, and bundle artifacts for download.
import pandas as pd, os
rd = f'{REPO}/reports/lap_time'
for fn in ('tft_breakdown.csv', 'tft_calibration.csv'):
    p = f'{rd}/{fn}'
    if os.path.exists(p):
        print('\n===', fn, '===')
        print(pd.read_csv(p).to_string(index=False))
# zip models + reports to /kaggle/working for one-click download (Output panel)
!cd $REPO && zip -qr /kaggle/working/tft_artifacts.zip models reports/lap_time
print('\nDownload: /kaggle/working/tft_artifacts.zip')